# Growth-Ignition Anatomy — what the starts of 3x runs looked like (blind study)
**The question (Jake, 2026-07-25):** map every S&P-name run of ≥3x within 2-3 years since 2015; extract the first 6 months;
dissect the ignition points with moving averages, oscillators, volume — vs a CONTROL group of random non-ignition dates.

**Token-free** (Yahoo v8, no key). Run top to bottom; the download cell takes ~4 min.

## ⚠️ Discipline header — read first
1. **Anatomy ≠ prediction.** Episodes are found by conditioning on a KNOWN future (the 3x happened). The output describes
   what ignitions looked like; it does not make matching states a buy signal.
2. **The base-rate cell (last) is the honest number**: profile-matching states went on to 3x ~18% of the time vs ~10%
   for random stock-days (first run 2026-07-25) — the profile DOUBLES the odds and still fails 82% of the time.
3. **Survivorship**: universe = today's S&P 500 (known winners) — inflates both the hit rate and the baseline.
4. First-run headline (2026-07-25, 573 ignitions/284 stocks): ignition = CAPITULATION, not breakout — median 38% below
   ATH, 22% BELOW the 200-SMA, RSI-14 ≈ 28, realized vol ~2x normal, volume ~120% of baseline; first 6 months median +41%
   with a −18% pullback inside; price reclaims the 200-SMA at median day 44. Crash-year cohorts (2018/20/22 = 57% of
   ignitions) are extreme versions; calm-year ignitions are milder (−8.5% vs 200-SMA, RSI ~31).

In [ ]:
# CELL 1 — parameters + universe
MULT = 3.0        # the run: >= 3x ...
H    = 756        # ... within this many trading days (~3 years)
SEP  = 365        # min days between two ignitions of the same stock
RANGE = '11y'

import requests, time, re, json
import pandas as pd, numpy as np
S = requests.Session(); S.headers['User-Agent'] = 'Mozilla/5.0 (research notebook)'
html = S.get('https://en.wikipedia.org/wiki/List_of_S%26P_500_companies', timeout=30).text
raw = sorted(set(re.findall(r'href="https?://(?:www\.)?(?:nyse|nasdaq|cboe)\.com[^"]*"[^>]*>([A-Z][A-Z0-9.\-]*)<', html)))
TICKERS = [t.replace('.', '-') for t in raw]
print(len(TICKERS), 'tickers')

In [ ]:
# CELL 2 — download daily adjusted close + volume (~4 min)
def pull(t):
    r = S.get(f'https://query1.finance.yahoo.com/v8/finance/chart/{t}',
              params={'range': RANGE, 'interval': '1d', 'events': 'div,split'}, timeout=20)
    j = r.json()['chart']['result'][0]
    q = j['indicators']['quote'][0]
    adj = j['indicators'].get('adjclose', [{}])[0].get('adjclose') or q['close']
    idx = pd.to_datetime(j['timestamp'], unit='s').normalize()
    a = pd.Series(adj, index=idx); v = pd.Series(q['volume'], index=idx)
    return a[~a.index.duplicated()], v[~v.index.duplicated()]

adj_cols, vol_cols, failed = {}, {}, []
for i, t in enumerate(TICKERS):
    try:
        adj_cols[t], vol_cols[t] = pull(t)
    except Exception:
        failed.append(t)
    if i % 100 == 0: print(i, end=' ', flush=True)
    time.sleep(0.1)
px = pd.DataFrame(adj_cols).sort_index(); vol = pd.DataFrame(vol_cols).sort_index()
print('\nmatrix:', px.shape, px.index[0].date(), '->', px.index[-1].date(), '| failed:', failed)

In [ ]:
# CELL 3 — indicators
sma20  = px.rolling(20).mean(); sma50 = px.rolling(50).mean(); sma200 = px.rolling(200).mean()
ath = px.cummax()
d = px.diff()
rsi = 100 - 100 / (1 + d.clip(lower=0).ewm(alpha=1/14, min_periods=14).mean()
                       / (-d.clip(upper=0)).ewm(alpha=1/14, min_periods=14).mean())
ret = px.pct_change()
rv63 = ret.rolling(63).std() * np.sqrt(252)
mom126 = px.pct_change(126)
slope200 = sma200.pct_change(20)
relvol = vol.rolling(21).mean() / vol.rolling(126).mean()
print('indicators ready')

In [ ]:
# CELL 4 — episode detection: ignition = the trough from which a >=3x-in-<=H-days run launched
episodes = []
for t in px.columns:
    p = px[t].dropna()
    if len(p) < H // 2: continue
    arr = p.values; n = len(arr)
    fwd = np.full(n, np.nan)
    for i in range(n - 1):
        j = min(n, i + 1 + H)
        if j > i + 1: fwd[i] = arr[i + 1:j].max()
    cand = (fwd / arr) >= MULT
    i = 0
    while i < n:
        if cand[i]:
            j = i
            while j + 1 < n and cand[j + 1]: j += 1
            k = i + int(np.argmin(arr[i:j + 1]))
            episodes.append((t, p.index[k], arr[k]))
            i = j + 1 + 126
        else:
            i += 1
eps = pd.DataFrame(episodes, columns=['tick', 'date', 'price']).sort_values(['tick', 'date'])
keep, last_by = [], {}
for _, r in eps.iterrows():
    if r['tick'] not in last_by or (r['date'] - last_by[r['tick']]).days > SEP:
        keep.append(r); last_by[r['tick']] = r['date']
eps = pd.DataFrame(keep)
print(f'ignitions: {len(eps)} across {eps.tick.nunique()} stocks')
print(eps.date.dt.year.value_counts().sort_index().to_string())

In [ ]:
# CELL 5 — anatomy at ignition vs CONTROL (random non-ignition stock-dates)
def feats(t, dt):
    try: i = px.index.get_loc(dt)
    except KeyError: return None
    def g(df):
        try: return df[t].iloc[i]
        except Exception: return np.nan
    p = g(px)
    out = dict(dd_ath=p/g(ath)-1, vs50=p/g(sma50)-1, vs200=p/g(sma200)-1, cross=g(sma50)/g(sma200)-1,
               rsi=g(rsi), rvol63=g(rv63), mom126=g(mom126), slope200=g(slope200), relvol=g(relvol))
    fut = px[t].iloc[i:i+126].dropna()
    if len(fut) > 60:
        out['ret6m'] = fut.iloc[-1]/fut.iloc[0]-1
        out['maxpull6m'] = (fut/fut.cummax()-1).min()
        rel = px[t].iloc[i:i+252] >= sma200[t].iloc[i:i+252]
        out['days_to_200'] = int(rel.values.argmax()) if rel.any() else 999
    return out

ig = pd.DataFrame([f for f in (feats(r.tick, r.date) for r in eps.itertuples()) if f])
rng = np.random.default_rng(42)
mes = px.index[252:][::21]
ig_by_tick = eps.groupby('tick')['date'].apply(list).to_dict()
ticks = list(px.columns); ctrl = []
while len(ctrl) < 3000:
    t = ticks[rng.integers(len(ticks))]; dt = mes[rng.integers(len(mes))]
    if any(abs((dt - x).days) < 365 for x in ig_by_tick.get(t, [])): continue
    f = feats(t, dt)
    if f and not np.isnan(f['vs200']): ctrl.append(f)
ct = pd.DataFrame(ctrl)

print('=== ignition vs control (median [IQR]) ===')
for k, lab in [('dd_ath','% below ATH'),('vs200','% vs 200-SMA'),('vs50','% vs 50-SMA'),('cross','50 vs 200 SMA'),
               ('rsi','RSI-14'),('rvol63','realized vol 63d'),('mom126','trailing 6-mo ret'),
               ('slope200','200-SMA slope'),('relvol','rel volume')]:
    a, b = ig[k].dropna(), ct[k].dropna()
    f = (lambda x: f'{x*100:6.1f}%') if k != 'rsi' else (lambda x: f'{x:6.1f}')
    print(f'{lab:20s} IGN {f(a.median())} [{f(a.quantile(.25))},{f(a.quantile(.75))}]  CTRL {f(b.median())} [{f(b.quantile(.25))},{f(b.quantile(.75))}]')
print('\n=== first 6 months after ignition ===')
for k, lab in [('ret6m','6-mo return'),('maxpull6m','worst pullback'),]:
    a = ig[k].dropna(); print(f'{lab:20s} median {a.median()*100:6.1f}%  [{a.quantile(.25)*100:6.1f}%, {a.quantile(.75)*100:6.1f}%]')
a = ig['days_to_200'].dropna(); a = a[a < 999]
print(f'days to reclaim 200-SMA: median {a.median():.0f}')

In [ ]:
# CELL 6 — THE HONEST NUMBER: base-rate of the ignition profile
# profile = RSI<35 AND >=15% below 200-SMA AND relvol>=1.1  (edit and re-poke from other angles)
state = (rsi < 35) & (px/sma200 - 1 <= -0.15) & (relvol >= 1.1)
hits = total = 0
for t in px.columns:
    s = state[t]; days = s[s.fillna(False)].index
    if len(days) == 0: continue
    p = px[t].dropna(); arr = p.values; n = len(arr)
    fwd = np.full(n, np.nan)
    for i in range(n - 1):
        j = min(n, i + 1 + H)
        if j > i + 1: fwd[i] = arr[i + 1:j].max()
    fs = pd.Series(fwd, index=p.index)
    for dt in days:
        if dt not in fs.index or dt > p.index[-253]: continue
        total += 1
        if fs[dt] / p[dt] >= MULT: hits += 1
print(f'profile state-days: {total};  went on to {MULT}x within {H}d: {hits} = {100*hits/max(total,1):.1f}%')
print('(first run: 18.0% vs ~9.8% for random stock-days — doubles the odds, still fails ~4 of 5 times)')

## V2 — unconditioned entries (Jake's correction: "they don't need to be bottoming stocks")
Entry = EARLIEST day from which the 3x completed (no trough-forcing), then split by starting state. First-run result
(2026-07-25): WASHOUT 63% / PULLBACK 21% / STRENGTH 16%. The STRENGTH cohort (VRT, CEG, LLY, KLAC, GEV...) is technically
INVISIBLE — RSI ~51, normal volume, mild uptrend = identical to control. Oscillators fingerprint only the washout cohort;
the from-strength compounders are found by theme/fundamentals, not by technical state.

In [ ]:
# CELL 7 — V2: entry = first buyable day (unconditioned), split by starting state
rows = []
for t in px.columns:
    p = px[t].dropna()
    if len(p) < H // 2: continue
    arr = p.values; n = len(arr)
    fwd = np.full(n, np.nan)
    for i in range(n - 1):
        j = min(n, i + 1 + H)
        if j > i + 1: fwd[i] = arr[i + 1:j].max()
    cand = (fwd / arr) >= MULT
    i = 0
    while i < n:
        if cand[i]:
            j = i
            while j + 1 < n and cand[j + 1]: j += 1
            rows.append((t, p.index[i]))          # ENTRY = first day of the stretch
            i = j + 1 + 126
        else:
            i += 1
v2 = pd.DataFrame(rows, columns=['tick', 'entry']).sort_values(['tick', 'entry'])
keep, last = [], {}
for _, r in v2.iterrows():
    if r['tick'] not in last or (r['entry'] - last[r['tick']]).days > SEP:
        keep.append(r); last[r['tick']] = r['entry']
v2 = pd.DataFrame(keep)

f2 = pd.DataFrame([dict(tick=r.tick, entry=r.entry, **f) for r in v2.itertuples()
                   if (f := feats(r.tick, r.entry))])
f2['cohort'] = np.where(f2['dd_ath'] >= -0.10, 'STRENGTH',
               np.where(f2['dd_ath'] >= -0.25, 'PULLBACK', 'WASHOUT'))
print('entry-state distribution:'); print(f2['cohort'].value_counts().to_string())
print(f"above 200-SMA at entry: {100*(f2['vs200']>0).mean():.0f}%")
for c, g in f2.groupby('cohort'):
    print(f'\n{c} (n={len(g)}) — medians')
    for k, lab in [('dd_ath','% off ATH'),('vs200','vs 200-SMA'),('rsi','RSI-14'),
                   ('relvol','rel volume'),('mom126','6-mo momentum'),('ret6m','next-6mo ret'),('maxpull6m','6mo max pull')]:
        v = g[k].dropna().median()
        print(f'  {lab:me 14s} {v:7.1f}'.replace('me ','') if k=='rsi' else f'  {lab:14s} {v*100:7.1f}%')
print('\nrecent STRENGTH entries:')
print(f2[f2.cohort=='STRENGTH'].sort_values('entry').tail(12)[['tick','entry']].to_string(index=False))